In [1]:
import pandas as pd
import sqlite3
%load_ext sql

In [2]:
df = pd.read_excel(r"D:\Healthcare-Operational-Analytics\Data\Staged & Cleaned Data\Hospital Dataset.xlsx", sheet_name = "Clean Data")

df.head()

,Patient_Id,Age,Age_Group,Gender,Blood_Type,City,State,Admission_Date,Length_Of_Stay_Days,Discharge_Date,...,Heart_Rate,Systolic_Bp,Diastolic_Bp,Temperature_F,Insurance_Provider,Billing_Amount_Usd,Payment_Status,Outcome,Readmitted_30_Days,Patient_Satisfaction_Score
0,PT109870,92.0,Senior,Female,O+,Jacksonville,FL,2023-09-22,7,2023-09-29,...,73.0,139.0,98.0,99.3,Unknown,4302217.0,Paid,Discharged - Recovered,False,4.0
1,PT104876,15.0,Teen,Male,A+,San Antonio,TX,2024-09-06,6,2024-09-12,...,94.0,145.0,89.0,101.2,Medicare,3795254.0,Partially Paid,Transferred,False,4.0
2,PT105484,17.0,Teen,Female,O+,Fort Worth,TX,2025-10-09,7,2025-10-16,...,72.0,125.0,78.0,97.3,United Healthcare,3303678.0,Paid,Discharged - Recovered,False,3.0
3,PT111789,23.0,Adult,Female,A+,Austin,TX,2023-04-16,5,2023-04-21,...,84.0,139.0,83.0,101.4,Medicare,3287499.0,Overdue,Discharged - Recovered,True,4.0
4,PT113751,17.0,Teen,Male,A+,Nyc,NY,2023-02-26,5,2023-03-03,...,76.0,119.0,61.0,99.0,Cigna,3051477.0,Pending,Discharged - Recovered,False,NaN


In [3]:
df.shape

(20000, 25)

In [4]:
conn = sqlite3.connect("hospital.db")

df.to_sql("hospital_data", conn, if_exists = "replace", index = False)

print("Database connected Successfully.")

Database connected Successfully.


In [5]:
print(df.columns.to_list())

['Patient_Id', 'Age', 'Age_Group', 'Gender', 'Blood_Type', 'City', 'State', 'Admission_Date', 'Length_Of_Stay_Days', 'Discharge_Date', 'Admission_Type', 'Department', 'Diagnosis', 'Attending_Doctor', 'Bed_Type', 'Heart_Rate', 'Systolic_Bp', 'Diastolic_Bp', 'Temperature_F', 'Insurance_Provider', 'Billing_Amount_Usd', 'Payment_Status', 'Outcome', 'Readmitted_30_Days', 'Patient_Satisfaction_Score']


In [6]:
%sql sqlite:///hospital.db

In [7]:
%%sql

SELECT COUNT(*) AS Total_patients
FROM hospital_data;

 * sqlite:///hospital.db
Done.


Total_patients
20000


Q1. Which hospital department treated the highest number of patients?

In [16]:
%%sql

SELECT department, COUNT(*) AS Patient_Count
FROM hospital_data
GROUP BY department
ORDER BY Patient_Count DESC;

 * sqlite:///hospital.db
Done.


Department,Patient_Count
Psychiatry,1413
Obstetrics,1389
Neurology,1372
Cardiology,1364
Urology,1344
Oncology,1337
Pulmonology,1321
General Surgery,1317
Nephrology,1316
Gastroenterology,1313


Q2. What is the average patient length of stay for each department? Sort from highest to lowest.

In [22]:
%%sql
SELECT Department, ROUND(AVG(Length_Of_Stay_Days),2) AS Average_length_of_stay
FROM hospital_data
GROUP BY department
ORDER BY Average_length_of_stay DESC;

 * sqlite:///hospital.db
Done.


Department,Average_length_of_stay
Neurology,5.94
Oncology,5.89
Nephrology,3.67
Pediatrics,3.6
Endocrinology,3.58
General Surgery,3.57
Pulmonology,3.56
Psychiatry,3.56
Dermatology,3.54
Orthopedics,3.53


Q3. Which five departments have the highest total billing amounts?

In [25]:
%%sql
SELECT Department, SUM(Billing_Amount_Usd) AS Total_billing_amount
FROM hospital_data
GROUP BY department
ORDER BY Total_billing_amount DESC
LIMIT 5;

 * sqlite:///hospital.db
Done.


Department,Total_billing_amount
Oncology,47792920.33
Neurology,39600854.69
Dermatology,28589209.42
Pediatrics,27150833.24
Nephrology,26073627.98


Q4. What percentage of patients were readmitted within 30 days for each hospital department?

In [39]:
%%sql

SELECT
    Department,
    COUNT(Readmitted_30_Days) AS Total_Valid_Records,
    ROUND(
        AVG(Readmitted_30_Days) * 100.0,
        2
    ) AS Readmission_Rate
FROM hospital_data
GROUP BY Department
ORDER BY Readmission_Rate DESC;

 * sqlite:///hospital.db
Done.


Department,Total_Valid_Records,Readmission_Rate
Emergency,235,19.15
Pulmonology,1292,16.56
Pediatrics,1288,16.15
Psychiatry,1384,16.11
General Surgery,1291,15.96
Urology,1312,15.93
Oncology,1313,15.92
Neurology,1336,15.64
Gastroenterology,1279,15.64
Emergency Medicine,1048,15.55


Q5. How many patient records belong to each payment status, and what is the total billing amount for each status?

In [44]:
%%sql

SELECT Payment_status, COUNT(*) AS Total_Patients, SUM(Billing_Amount_Usd) AS Total_billing_amount
FROM hospital_data
GROUP BY Payment_status
ORDER BY Total_Patients DESC;

 * sqlite:///hospital.db
Done.


Payment_Status,Total_Patients,Total_billing_amount
Paid,8837,154729520.72
Pending,3910,68680745.6
Insurance Processing,3883,63813433.75
Overdue,1541,36598766.24
Partially Paid,1328,28899323.87
Unknown,501,10431371.23


Q6. Which five hospital departments have the lowest average patient satisfaction scores?

In [ ]:
%%sql

SELECT 
     Department,
     ROUND(AVG(Patient_Satisfaction_Score),2) 
        AS Avg_Patient_Satisfaction_score
FROM hospital_data
WHERE Patient_Satisfaction_Score BETWEEN 1 AND 10
GROUP By Department
ORDER BY AVG_Patient_Satisfaction_Score ASC
LIMIT 5;

 * sqlite:///hospital.db
Done.


Department,Avg_Patient_Satisfaction_score
Emergency Medicine,3.91
Obstetrics,3.91
Urology,3.92
Gastroenterology,3.93
Endocrinology,3.94


Q7. How many patients were admitted each month, and which months had the highest admission volume?

In [61]:
%%sql

SELECT strftime('%Y-%m', Admission_date) AS Admission_Month, COUNT(*) AS Total_Patients
FROM hospital_data
GROUP BY Admission_Month
ORDER BY Total_Patients DESC;

 * sqlite:///hospital.db
Done.


Admission_Month,Total_Patients
2022-05,469
2022-10,447
2025-05,444
2024-08,441
2024-12,438
2024-07,437
2023-07,437
2023-11,436
2024-01,433
2025-11,431


Q8. Which departments generate the highest total billing amounts.

In [72]:
%%sql
WITH Billing_Amount AS (
    SELECT
        Department,
        SUM(Billing_Amount_Usd) as Amount
    FROM hospital_data
    GROUP BY Department
)
SELECT 
            Department,
            Amount AS Total_Billing,
            DENSE_RANK() OVER(
                ORDER BY Amount DESC
            ) AS Billing_Rank
            FROM Billing_Amount
            ORDER BY Billing_Rank;

 * sqlite:///hospital.db
Done.


Department,Total_Billing,Billing_Rank
Oncology,47792920.33,1
Neurology,39600854.69,2
Dermatology,28589209.42,3
Pediatrics,27150833.24,4
Nephrology,26073627.98,5
Psychiatry,25195557.41,6
General Surgery,24659720.69,7
Pulmonology,22548689.09,8
Endocrinology,22021575.95,9
Gastroenterology,20917499.11,10


Q9. Identify patients who stayed longer than the average patient in their respective departments.

In [78]:
%%sql

SELECT
    h1.Patient_Id,
    h1.Department,
    h1.Length_Of_Stay_Days
FROM hospital_data AS h1
WHERE h1.Length_Of_Stay_Days > (
    SELECT AVG(h2.Length_Of_Stay_Days)
    FROM hospital_data AS h2
    WHERE h2.Department = h1.Department
)
ORDER BY h1.Length_Of_Stay_Days DESC;

 * sqlite:///hospital.db
Done.


Patient_Id,Department,Length_Of_Stay_Days
PT102449,Oncology,16
PT100937,Neurology,15
PT117501,Oncology,15
PT117404,Neurology,15
PT112096,Neurology,15
PT103925,Neurology,14
PT113831,Oncology,14
PT106219,Oncology,14
PT119927,Neurology,14
PT101536,Oncology,14


Q10. Monthly Billing Growth
Calculate monthly total hospital billing and the percentage change compared with the previous month.

In [84]:
%%sql

WITH Monthly_Billing AS (
    SELECT
        strftime('%Y-%m', Admission_Date) AS Month,
        SUM(Billing_Amount_Usd) AS Total_Billing
    FROM hospital_data
    WHERE strftime('%Y-%m', Admission_Date) IS NOT NULL
    GROUP BY strftime('%Y-%m', Admission_Date)
),
Billing_Comparison AS (
    SELECT
        Month,
        Total_Billing,
        LAG(Total_Billing) OVER (
            ORDER BY Month
        ) AS Previous_Month_Billing
    FROM Monthly_Billing
)
SELECT
    Month,
    ROUND(Total_Billing, 2) AS Total_Billing,
    ROUND(Previous_Month_Billing, 2) AS Previous_Month_Billing,
    ROUND(
        (Total_Billing - Previous_Month_Billing)
        * 100.0 /
        NULLIF(Previous_Month_Billing, 0),
        2
    ) AS Growth_Percentage
FROM Billing_Comparison
ORDER BY Month;

 * sqlite:///hospital.db
Done.


Month,Total_Billing,Previous_Month_Billing,Growth_Percentage
2022-01,7333046.08,None,None
2022-02,5681793.7,7333046.08,-22.52
2022-03,7682746.27,5681793.7,35.22
2022-04,6823889.54,7682746.27,-11.18
2022-05,11520516.7,6823889.54,68.83
2022-06,4235494.61,11520516.7,-63.24
2022-07,9263922.32,4235494.61,118.72
2022-08,8280654.79,9263922.32,-10.61
2022-09,7675786.09,8280654.79,-7.3
2022-10,9230958.57,7675786.09,20.26
